# Docker

In this demo we are going to go through the process of utilizing Docker as part of the Model Deployment process.

在本演示中，我们将介绍如何将 Docker 用作模型部署过程的一部分。

We will cover Docker basics, build an image with our containing our flask app, push it to a repo, and finally run a container from the image.

我们将涵盖 Docker 基础知识，构建包含 Flask 应用的镜像，将其推送到仓库，最后从镜像运行容器。

## Installing Docker | 安装 Docker

You can install the Docker engine on Linux, Windows and Mac.

您可以在 Linux、Windows 和 Mac 上安装 Docker 引擎。

Below steps uses Ubuntu using `apt`.

以下步骤使用 `apt` 在 Ubuntu 上安装。

### First we need to add the official GPG Key | 首先添加官方 GPG 密钥

```bash
apt-get update
apt-get install ca-certificates curl
install -m 0755 -d /etc/apt/keyrings
curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
chmod a+r /etc/apt/keyrings/docker.asc
```

### Next we need to add the repository to Apt sources. | 接下来添加仓库到 Apt 源

```bash
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  
  tee /etc/apt/sources.list.d/docker.list > /dev/null
```


### Run the installation of the CLI and the Engine | 运行 CLI 和引擎的安装

```bash
apt-get update
apt-get install docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin -y
```

### Start and enable the service | 启动并启用服务

```bash
systemctl start docker.service
systemctl enable docker.service
```

### Verify the installation | 验证安装

```bash
docker version
docker run hello-world
```

## Building a Docker image | 构建 Docker 镜像

Building a Docker image involves using the `docker build` command to package an application and its dependencies into a portable template called an image. 

构建 Docker 镜像涉及使用 `docker build` 命令将应用程序及其依赖项打包到一个称为镜像的可移植模板中。

This process uses a Dockerfile, which are instructions to define the image's contents and configuration.

此过程使用 Dockerfile，其中包含定义镜像内容和配置的指令。

In the same path as the Dockerfile.

在与 Dockerfile 相同的路径中。

```bash 
docker build -t mobilenetv3lg-flask:v1.0 .
```

NOTE: the size of the image on initial build.

注意：初始构建时镜像的大小。

## Creating a Dockerfile | 创建 Dockerfile

Creating a Dockerfile involves writing a simple text file that defines the steps to package an application into a Docker image. 

创建 Dockerfile 涉及编写一个简单的文本文件，该文件定义了将应用程序打包到 Docker 镜像中的步骤。

It includes instructions such as the base image to use, dependencies to install, files to include, and the commands to run the application.

它包括一些指令，例如要使用的基础镜像、要安装的依赖项、要包含的文件以及运行应用程序的命令。

```bash
# Use official Python base image | 使用官方 Python 基础镜像
FROM python:3.11-slim

# Set the working directory | 设置工作目录
WORKDIR /opt/app

# Copy requirements and install dependencies | 复制 requirements 并安装依赖项
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app code | 复制应用代码
COPY ./flask_app .

# Create a user and group for running the app | 创建用于运行应用的用户和组
RUN groupadd -r pytorch && useradd --no-log-init -r -g pytorch pytorch

# Change ownership of the home directory | 更改主目录的所有权
RUN chown -R pytorch:pytorch /home/pytorch

# Change ownership of the app directory | 更改应用目录的所有权
RUN chown -R pytorch:pytorch /opt/app

# Switch to the created user | 切换到创建的用户
USER pytorch

# Expose the port for Flask | 公开 Flask 的端口
EXPOSE 8000

# Command to run the app | 运行应用的命令
CMD ["flask", "run", "--host=0.0.0.0", "--port=8000"]

```

 ## Running a Container from an Image | 从镜像运行容器

Once we have our model built we can run a container from our model using the `docker run` command.

一旦我们构建了模型，我们就可以使用 `docker run` 命令从我们的模型运行容器。

```bash
docker run -p 8000:8000 mobilenetv3lg-flask:v1.0
```

NOTE: add a `-d` command makes it run in daemon mode where it starts and runs until killed but docker CLI.

注意：添加 `-d` 命令使其以守护进程模式运行，它将启动并运行直到被 docker CLI 终止。

In [ ]:
# Test a request | 测试请求
import requests
import base64

# Open image and encode it | 打开图像并编码
with open('dog-1.jpg', 'rb') as img_file:
    base64_string = base64.b64encode(img_file.read()).decode('utf-8')

# JSON payload with the Base64 encoded image | 包含 Base64 编码图像的 JSON 负载
payload = {
    "image": base64_string
}

# Set the headers | 设置头部
headers = {
    "Content-Type": "application/json"
}

# Send POST request | 发送 POST 请求
response = requests.post("http://127.0.0.1:8000/predict", 
                         json=payload, 
                         headers=headers)

# Print the response | 打印响应
print("Response JSON:", response.json())

In [ ]:
import json 

# Downloaded from Hugging Face | 从 Hugging Face 下载
# https://huggingface.co/datasets/huggingface/label-files/blob/main/imagenet-1k-id2label.json
with open("labels.json", "r") as f:
    imagenet_classes = json.load(f)

# Get the actual class name from our labels | 从标签中获取实际的类名
print(imagenet_classes['207'])

## Push a Docker Image to a Registry | 将 Docker 镜像推送到仓库

Once you have tested your image, you can push it to a registry for others to use. 

测试镜像后，您可以将其推送到仓库供他人使用。

Pushing an image to a Docker registry involves tagging the image with the registry name and using the docker push command to upload it.

将镜像推送到 Docker 仓库涉及使用仓库名称标记镜像，并使用 docker push 命令上传它。

```bash
docker tag mobilenetv3lg-flask:v1.0 username/mobilenetv3lg-flask:v1.0
```

Then push it.

然后推送它。

```bash
docker push username/mobilenetv3lg-flask:v1.0
```

NOTE: You must be authenticated before logging in. You can use `docker login` command for docker registry.

注意：您必须在登录之前进行身份验证。您可以使用 `docker login` 命令登录 docker 仓库。

## Pulling an Image | 拉取镜像

Once the image lives on a registry, anyone with access can pull the image and create a container from it.

一旦镜像位于仓库中，任何有权限的人都可以拉取镜像并从中创建容器。

```bash
docker pull username/mobilenetv3lg-flask:v1.0
```

```bash
docker run -p 8000:8000 username/mobilenetv3lg-flask:v1.0
```